# nb13 · Multi-seed data-efficiency reruns (TRAINING error bars)

Same finetune sweep as nb05, but each budget is trained **`len(TRAIN_SEEDS)` times** with different
training seeds (data held fixed per budget) → real **training-run-to-run** error bars on `tone_i2`,
the thing that decides whether 5 h truly differs from 15 h (the reviewer's #1 fix). Decode is held at
a single `EVAL_SEED` so the spread reflects *training*, not resynthesis noise.

**GPU:** A100-80GB (training). **Time:** ~2.5–3 h for budgets `[1,5,15] h × 3 seeds` (≈1.42 it/s →
15 h≈1.4k steps≈16 min/run, 5 h≈5 min, 1 h≈1 min; + Stage-0 tokenize + scoring). Add the 28.5 h budget
or 5 seeds and it grows ~linearly (5 seeds ≈ 4–5 h; +28.5 h budget ≈ +2 h).

Checkpoints are **not** uploaded (scores only). Results → `…/omnivoice_ablation/data_efficiency_multiseed.v1.json`.

## 1. Install + restart

In [ ]:
%pip install --quiet "numpy<2"
%pip install --quiet omnivoice
%pip install --quiet boto3 soundfile soxr librosa speechbrain "swift-f0" pyworld accelerate safetensors webdataset "huggingface_hub>=0.24.0" tqdm matplotlib
%pip uninstall -y hf-xet
import os
print("Installs done. RESTARTING so numpy<2 takes effect — run from the NEXT cell.", flush=True)
os._exit(0)

In [ ]:
import numpy as np
assert np.__version__.startswith("1."), "numpy<2 pin did not take — re-run install + restart"
print("numpy", np.__version__, "OK")

## 2. Clone OmniVoice (+ introspect finetune surface) + `mosesdaudu001/tone-on-a-budget` (approachB gate)

In [ ]:
import os, subprocess, sys, shutil, json
OV_DIR = "/content/OmniVoice" if os.path.isdir("/content") else "/tmp/OmniVoice"
if os.path.exists(OV_DIR): shutil.rmtree(OV_DIR)
subprocess.run(["git","clone","--depth","1","https://github.com/k2-fsa/OmniVoice.git",OV_DIR],check=True)
OV_EX=os.path.join(OV_DIR,"examples"); sys.path.insert(0,OV_DIR)
assert os.path.exists(os.path.join(OV_EX,"run_finetune.sh")), "MISSING run_finetune.sh — recon stale"
for mod in ["omnivoice.scripts.extract_audio_tokens","omnivoice.cli.train"]:
    r=subprocess.run([sys.executable,"-c",f"import importlib;importlib.import_module('{mod}')"],
                     capture_output=True,text=True,cwd=OV_DIR,env=dict(os.environ,PYTHONPATH=OV_DIR))
    assert r.returncode==0, f"{mod} not importable:\n{r.stderr[-600:]}"
STOCK=json.load(open(os.path.join(OV_EX,"config","train_config_finetune.json")))
assert STOCK.get("init_from_checkpoint")=="k2-fsa/OmniVoice" and STOCK.get("num_audio_codebook")==8
# Yoruba runtime code must be 'yo'
yo_ok=False
for ln in open(os.path.join(OV_DIR,"docs","lang_id_name_map.tsv")):
    c=ln.rstrip("\n").split("\t")
    if c and c[0]=="yo": yo_ok=True
assert yo_ok, "Yoruba code is not 'yo'"
print("finetune surface OK | stock steps", STOCK.get("steps"), "| codebooks", STOCK.get("num_audio_codebook"))

In [ ]:
# tone-metric package (public) — replaces the old git clone + sys.path of tone_metric/
%pip install --quiet --no-deps "git+https://github.com/mosesdaudu001/tone-on-a-budget.git"
import os, subprocess, sys, shutil, importlib
importlib.invalidate_caches()
import tone_metric
from tone_metric import tone_probe as tp, tone_f0_abs as f0a
from tone_metric.grpo_reward import RewardModels
CODE_DIR = os.path.dirname(tone_metric.__file__)   # minimal_pairs_draft.json ships as package data
print("tone_metric", tone_metric.__version__, "from", CODE_DIR)


## 3. Secrets + S3 + constants + Xet env + pre-fetch base snapshot + sweep knobs

In [ ]:
import os, getpass, boto3, random, torch, numpy as np, shutil
def _secret(k):
    try:
        from google.colab import userdata
        v=userdata.get(k)
        if v: return v
    except Exception: pass
    return os.environ.get(k) or getpass.getpass(f"{k}: ")
os.environ["AWS_ACCESS_KEY_ID"]=_secret("AWS_ACCESS_KEY_ID")
os.environ["AWS_SECRET_ACCESS_KEY"]=_secret("AWS_SECRET_ACCESS_KEY")
os.environ["AWS_DEFAULT_REGION"]=os.environ.get("AWS_DEFAULT_REGION","us-east-1")
HF_TOKEN=_secret("HF_TOKEN"); os.environ["HF_TOKEN"]=HF_TOKEN or ""
if HF_TOKEN:
    from huggingface_hub import login; login(token=HF_TOKEN)
os.environ["HF_HUB_DISABLE_XET"]="1"; os.environ["HF_HUB_ENABLE_HF_TRANSFER"]="0"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"]="60"; os.environ["HF_HUB_ETAG_TIMEOUT"]="60"
try:
    import huggingface_hub.constants as _hc; _hc.HF_HUB_DISABLE_XET=True
except Exception: pass
BUCKET="codec-audio-data"
s3=boto3.client("s3",region_name=os.environ["AWS_DEFAULT_REGION"]); s3.head_bucket(Bucket=BUCKET)
DEVICE="cuda" if torch.cuda.is_available() else "cpu"
assert DEVICE=="cuda", "the sweep trains — needs the A100"
SEED=4242; SR=24000; LANG_CODE="yo"
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# ===================== SWEEP KNOBS (no edits needed) =====================
SWEEP_BUDGETS  = [15.0, 5.0, 1.0]   # curve-defining budgets (add 28.5 for full; 0h = zero-shot 0.598)
TRAIN_SEEDS    = [4242, 1234, 2026] # >=3 training reruns/budget -> TRAINING error bars (reviewer fix)
EVAL_SEED      = 99                 # FIXED decode across all models -> spread reflects training, not decode
EPOCHS_TARGET  = 10                 # full saturated by ~ep 9
CLIPS_PER_STEP = 41.3               # calibrated: 10731 clips / ~260 steps-per-epoch @ batch_tokens=16384
BATCH_TOKENS   = 16384
DEV_HOURS      = 0.3
N_LONG_EVAL    = 8
# ========================================================================

BIBLE_MANIFEST="tts_data/yoruba_gold/s1_train.v2.jsonl"
HOLDOUTS_KEY="tts_data/yoruba_gold/holdouts.v1.json"
S3_TONE_PREFIX="tts_data/yoruba/tone_v2"; F0CAL_KEY=f"{S3_TONE_PREFIX}/f0_abs_calibration.v1.json"
ABLATION_KEY="tts_data/yoruba/omnivoice_ablation/data_efficiency.v1.json"
MULTISEED_KEY="tts_data/yoruba/omnivoice_ablation/data_efficiency_multiseed.v1.json"
WORK="/content/ov_sweep" if os.path.isdir("/content") else "/tmp/ov_sweep"
LOCAL_WAVS=os.path.join(WORK,"wavs"); os.makedirs(LOCAL_WAVS,exist_ok=True)

from huggingface_hub import snapshot_download
HUB=os.path.join(os.path.expanduser(os.environ.get("HF_HOME","~/.cache/huggingface")),"hub")
if os.path.isdir(os.path.join(HUB,".locks")): shutil.rmtree(os.path.join(HUB,".locks"),ignore_errors=True)
OV_BASE=None
for _k in range(3):
    try: OV_BASE=snapshot_download("k2-fsa/OmniVoice",max_workers=1,etag_timeout=60); break
    except Exception as e: print("base prefetch retry:",type(e).__name__,e)
for _k in range(3):
    try: snapshot_download("eustlb/higgs-audio-v2-tokenizer",max_workers=1,etag_timeout=60); break   # Stage-0 codec
    except Exception as e: print("codec prefetch retry:",type(e).__name__,e)
assert OV_BASE and os.path.isdir(os.path.join(OV_BASE,"audio_tokenizer")), "base prefetch failed"
print("base snapshot:",OV_BASE,"| budgets:",SWEEP_BUDGETS,"| epochs_target",EPOCHS_TARGET)

## 4. Load the clean corpus once (shuffled), download the LARGEST-budget wavs once

Budgets are **prefixes of one shared deterministic shuffle**, so 5 h ⊂ 15 h and 1 h ⊂ 5 h — we download
the 15 h wav set (+ dev) a single time and every budget reuses it.

In [ ]:
import io, json, unicodedata
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm
def norm_key(s): return " ".join(unicodedata.normalize("NFC",s).lower().split())
HELD=set()
hold=json.loads(s3.get_object(Bucket=BUCKET,Key=HOLDOUTS_KEY)["Body"].read())
def _collect(x):
    if isinstance(x,str): HELD.add(norm_key(x))
    elif isinstance(x,dict): [_collect(v) for v in x.values()]
    elif isinstance(x,list): [_collect(v) for v in x]
for k in ("eval_texts","minimal_pairs"):
    if k in hold: _collect(hold[k])
cands=[]
for raw in io.TextIOWrapper(s3.get_object(Bucket=BUCKET,Key=BIBLE_MANIFEST)["Body"],encoding="utf-8"):
    r=json.loads(raw)
    if norm_key(r["text"]) in HELD: continue
    cands.append(dict(id=r["id"],text=r["text"],dur=float(r["duration_sec"]),wav_key=r["audio_s3_key"]))
rng=random.Random(SEED); rng.shuffle(cands)
def take_hours(rows,budget_h):
    if budget_h is None: return rows,sum(c["dur"] for c in rows)
    acc,out=0.0,[]
    for c in rows:
        if acc>=budget_h*3600: break
        out.append(c); acc+=c["dur"]
    return out,acc
dev_rows,dev_s=take_hours(cands,DEV_HOURS); rest=cands[len(dev_rows):]
print(f"candidates {len(cands)} ({sum(c['dur'] for c in cands)/3600:.2f} h) | dev {len(dev_rows)} | rest {len(rest)}")
# download wavs for the LARGEST budget + dev, once
largest=max(SWEEP_BUDGETS)
big_train,_=take_hours(rest,largest)
need=big_train+dev_rows
def _dl(c):
    local=os.path.join(LOCAL_WAVS,c["id"]+".wav")
    if not os.path.exists(local): s3.download_file(BUCKET,c["wav_key"],local)
    c["audio_path"]=local; return c
with ThreadPoolExecutor(max_workers=32) as ex:
    list(tqdm(ex.map(_dl,need),total=len(need),desc=f"download wavs (<= {largest:g}h + dev)"))
ID2PATH={c["id"]:c["audio_path"] for c in need}
print("wavs local:",len(ID2PATH))

## 5. Gate stack + fixed eval set (loaded once, reused by every budget)

In [ ]:
import json, soundfile as sf, numpy as np, random
from transformers import AutoModel, AutoFeatureExtractor
rm=RewardModels(device=DEVICE); assert rm.ecapa is not None
enc=AutoModel.from_pretrained("ajesujoba/AfriHuBERT").to(DEVICE).eval()
fe =AutoFeatureExtractor.from_pretrained("ajesujoba/AfriHuBERT")
objs=s3.list_objects_v2(Bucket=BUCKET,Prefix=f"{S3_TONE_PREFIX}/tone_probe_L").get("Contents")
probe_key=sorted(o["Key"] for o in objs)[-1]; s3.download_file(BUCKET,probe_key,f"{WORK}/tone_probe.pt")
PROBE,PMETA=tp.load_probe(f"{WORK}/tone_probe.pt",DEVICE); LAYER=PMETA.get("layer",9)
s3.download_file(BUCKET,F0CAL_KEY,f"{WORK}/f0cal.json"); F0CAL=json.load(open(f"{WORK}/f0cal.json"))
I2_TH,I2_TL=F0CAL["theta_h"],F0CAL["theta_l"]; I2_MODE,I2_LATE=F0CAL.get("mode","blind"),F0CAL.get("late_frac",0.5)
# fixed probe set (minimal pairs + held-out long lines + one bible ref) — identical across budgets
MP=json.load(open(os.path.join(CODE_DIR,"minimal_pairs_draft.json"))); CARRIER=MP["carriers"][0]["template"]
probe_lines=[]
for s_ in MP["sets"]:
    for j,it in enumerate(s_["items"]):
        probe_lines.append(dict(id=f"mp_{s_['base']}_{j}",kind="minpair",text=CARRIER.replace("___",it["text"])))
EVAL_ALL=list(hold["eval_texts"]); _r=random.Random(SEED)
for k,e in enumerate(_r.sample(EVAL_ALL,N_LONG_EVAL)):
    probe_lines.append(dict(id=f"long_{k}_{e['base']}",kind="long",text=e["text"]))
ref_row=None
for raw in io.TextIOWrapper(s3.get_object(Bucket=BUCKET,Key=BIBLE_MANIFEST)["Body"],encoding="utf-8"):
    r=json.loads(raw)
    if r.get("source")=="bible" and 3.0<=float(r.get("duration_sec",0))<=10.0: ref_row=r; break
REF_WAV_PATH=f"{WORK}/ref.wav"; s3.download_file(BUCKET,ref_row["audio_s3_key"],REF_WAV_PATH)
REF_TEXT=ref_row["text"]; ref_wav,ref_sr=sf.read(REF_WAV_PATH,dtype="float32")
if ref_wav.ndim>1: ref_wav=ref_wav.mean(-1)
print("gate ready | probe lines",len(probe_lines))

## 6. Helpers + `run_one_budget(hours)` — the whole pipeline for one curve point

Defines everything; the budget **sections** below just call it. Stage-1 streams the FULL log to S3
(`.../sweep_logs/train_<tag>.log`) and prints only the per-100-step summary, so Colab's 5000-line cap
never loses the loss curve.

In [ ]:
import os, json, subprocess, sys, glob, datetime, shutil
import soundfile as sf, numpy as np
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm
import IPython.display as ipd, torch
from omnivoice import OmniVoice

AUX=["tokenizer.json","tokenizer_config.json","special_tokens_map.json","vocab.json","merges.txt",
     "chat_template.jinja","generation_config.json","added_tokens.json"]
def materialize_ckpt(d):
    have=set(os.listdir(d))
    for fn in AUX:
        if fn not in have and os.path.exists(os.path.join(OV_BASE,fn)): shutil.copy(os.path.join(OV_BASE,fn),os.path.join(d,fn))
    if not os.path.isdir(os.path.join(d,"audio_tokenizer")): shutil.copytree(os.path.join(OV_BASE,"audio_tokenizer"),os.path.join(d,"audio_tokenizer"))
    return d

def write_manifest(rows,path):
    with open(path,"w",encoding="utf-8") as f:
        for c in rows:
            f.write(json.dumps({"id":c["id"],"audio_path":ID2PATH[c["id"]],"text":c["text"],
                                "language_id":LANG_CODE,"duration":round(c["dur"],3)},ensure_ascii=False)+"\n")

def stage0(split,split_jsonl,tag,token_dir):
    out_tar=os.path.join(token_dir,split,"audios","shard-%06d.tar")
    out_jsonl=os.path.join(token_dir,split,"txts","shard-%06d.jsonl")
    os.makedirs(os.path.dirname(out_tar),exist_ok=True); os.makedirs(os.path.dirname(out_jsonl),exist_ok=True)
    cmd=[sys.executable,"-m","omnivoice.scripts.extract_audio_tokens","--input_jsonl",split_jsonl,
         "--tar_output_pattern",out_tar,"--jsonl_output_pattern",out_jsonl,
         "--tokenizer_path","eustlb/higgs-audio-v2-tokenizer","--nj_per_gpu","3","--shuffle","True"]
    subprocess.run(cmd,check=True,cwd=OV_DIR,env=dict(os.environ,CUDA_VISIBLE_DEVICES="0",PYTHONPATH=OV_DIR))
    lst=os.path.join(token_dir,split,"data.lst"); assert os.path.exists(lst), f"Stage0 no {lst}"
    return lst

def build_cfg(steps,train_lst,dev_lst,tag,output_dir,seed=SEED):
    save_eval=max(50,steps//8)
    cfg=dict(STOCK); cfg.update({
        "init_from_checkpoint":"k2-fsa/OmniVoice","resume_from_checkpoint":None,"attn_implementation":"sdpa",
        "learning_rate":1e-5,"steps":int(steps),"warmup_type":"ratio","warmup_ratio":0.01,"warmup_steps":0,
        "batch_tokens":BATCH_TOKENS,"gradient_accumulation_steps":1,"mixed_precision":"bf16","allow_tf32":True,
        "num_workers":2,"logging_steps":max(10,save_eval//5),"eval_steps":save_eval,"save_steps":int(steps),   # save ONLY at the end
        "keep_last_n_checkpoints":1,"seed":seed,"language_ratio":1.0})
    for k,v in {"max_sample_tokens":2000,"min_sample_tokens":50,"max_batch_size":64}.items(): cfg.setdefault(k,v)
    data_cfg={"train":[{"manifest_path":[train_lst]}],"dev":[{"manifest_path":[dev_lst]}]}
    cp=os.path.join(WORK,f"train_config_{tag}.json"); dp=os.path.join(WORK,f"data_config_{tag}.json")
    json.dump(cfg,open(cp,"w"),indent=2); json.dump(data_cfg,open(dp,"w"),indent=2)
    return cp,dp,cfg

def stage1(cp,dp,output_dir,tag):
    cmd=["accelerate","launch","--gpu_ids","0","--num_processes","1","-m","omnivoice.cli.train",
         "--train_config",cp,"--data_config",dp,"--output_dir",output_dir]
    print(">>"," ".join(cmd))
    logf=os.path.join(WORK,f"train_{tag}.log")
    proc=subprocess.Popen(cmd,cwd=OV_DIR,env=dict(os.environ,PYTHONPATH=OV_DIR),
                          stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    with open(logf,"w") as lf:
        for line in proc.stdout:
            lf.write(line)
            low=line.lower()
            if ("train/loss" in line) or ("eval loss" in low) or ("saved checkpoint" in low) \
               or ("epoch" in low and "starting" in low) or ("error" in low) or ("traceback" in low):
                print(line,end="")
    rc=proc.wait()
    s3.upload_file(logf,BUCKET,f"tts_data/yoruba/sweep_logs/train_{tag}.log")   # FULL log durable
    assert rc==0, f"training exited {rc} — see s3 sweep_logs/train_{tag}.log"
    ck=sorted(glob.glob(os.path.join(output_dir,"checkpoint-*")),key=lambda p:int(p.rsplit("-",1)[-1]))
    assert ck, "no checkpoint written"; return ck[-1]

def _bal(pairs):
    if not pairs: return float("nan")
    tot,cor=defaultdict(int),defaultdict(int)
    for pp,tt in pairs: tot[tt]+=1; cor[tt]+=int(pp==tt)
    rs=[cor[c]/tot[c] for c in tot if tot[c]>0]; return float(np.mean(rs)) if rs else float("nan")
def score_clip(wav,fs,text):
    logits,n16=rm.asr_logits(wav,fs); cer=RewardModels.cer(text,rm.decode_logits(logits))
    i1=tp.probe_score(wav,fs,text,PROBE,enc,fe,asr=rm.asr,proc=rm.asr_proc,device=DEVICE,emissions=logits,n16=n16,layer=LAYER)
    i2=f0a.f0_abs_score(wav,fs,text,asr=rm.asr,proc=rm.asr_proc,device=DEVICE,emissions=logits,n16=n16,
                        theta_h=I2_TH,theta_l=I2_TL,mode=I2_MODE,mid_ref=None,late_frac=I2_LATE)
    pairs=[(pp,tt) for pp,tt in zip(i2["pred"],i2["target"]) if pp is not None]
    return dict(cer=cer,i1_acc=i1["accuracy"],i1_cov=i1["coverage"],tone_i2_cov=i2["coverage"],
                ssim=rm.ssim(wav,fs,ref_wav,ref_sr),len_ratio=(len(wav)/fs)/max(0.157*len(text),1e-6),pairs=pairs,wav=wav)
def eval_budget(final_ckpt,tag,hours):
    m=OmniVoice.from_pretrained(materialize_ckpt(final_ckpt),device_map="cuda:0",dtype=torch.float16)
    torch.manual_seed(EVAL_SEED)   # fixed decode across all models
    rows=[]
    for p in tqdm(probe_lines,desc=f"eval[{tag}]"):
        a=m.generate(text=p["text"],language=LANG_CODE,ref_audio=REF_WAV_PATH,ref_text=REF_TEXT)
        rows.append(dict(kind=p["kind"],text=p["text"],**score_clip(np.asarray(a[0],dtype="float32"),SR,p["text"])))
    allp=[pr for r in rows for pr in r["pairs"]]; v=lambda k:[r[k] for r in rows if r[k]==r[k]]
    AGG=dict(n=len(rows),tone_i2=_bal(allp),cer=float(np.mean(v("cer"))),ssim=float(np.mean(v("ssim"))),
             i1_acc=float(np.mean(v("i1_acc"))),tone_i2_cov=float(np.mean([r["tone_i2_cov"] for r in rows])),
             len_ratio=float(np.mean([r["len_ratio"] for r in rows])))
    print(f"[{tag}] AGG:",json.dumps({k:round(x,3) if isinstance(x,float) else x for k,x in AGG.items()}))
    # upload eval wavs + inline-play a few (clone of the bible ref)
    pre=f"tts_data/yoruba/omnivoice_ft_probe/{tag}"
    with ThreadPoolExecutor(max_workers=16) as ex:
        def _up(r):
            loc=f"{WORK}/ev_{tag}_{r['kind']}_{abs(hash(r['text']))%99999}.wav"; sf.write(loc,r["wav"],SR)
            s3.upload_file(loc,BUCKET,f"{pre}/wav/{os.path.basename(loc)}"); return loc
        list(ex.map(_up,rows))
    print(f"  eval wavs -> s3://{BUCKET}/{pre}/wav/  | inline samples:")
    for r in ([x for x in rows if x["kind"]=="minpair"][:3]+[x for x in rows if x["kind"]=="long"][:1]):
        print("  ",r["text"][:46]); ipd.display(ipd.Audio(r["wav"],rate=SR))
    del m;
    import gc; gc.collect(); torch.cuda.empty_cache()
    return AGG

def append_table(tag,hours,train_hours,clips,steps,AGG):
    try: TABLE=json.loads(s3.get_object(Bucket=BUCKET,Key=ABLATION_KEY)["Body"].read())
    except Exception:
        TABLE={"meta":{"axis":"data-efficiency (clean Yoruba hours -> tone_i2)","model":"k2-fsa/OmniVoice finetune",
                       "protocol":f"fix-epochs={EPOCHS_TARGET} @ batch_tokens={BATCH_TOKENS}","seed":SEED},"runs":{}}
    TABLE["runs"][tag]={"hours_budget":hours,"train_hours_actual":round(train_hours,3),"train_clips":clips,
        "steps":steps,"epochs_target":EPOCHS_TARGET,"lr":1e-5,"batch_tokens":BATCH_TOKENS,
        "tone_i2":AGG["tone_i2"],"cer":AGG["cer"],"ssim":AGG["ssim"],"i1_acc":AGG["i1_acc"],
        "len_ratio":AGG["len_ratio"],"date":datetime.date.today().isoformat(),
        "ckpt_s3":f"s3://{BUCKET}/tts_checkpoints/omnivoice_yoruba/{tag}/"}
    s3.put_object(Bucket=BUCKET,Key=ABLATION_KEY,Body=json.dumps(TABLE,ensure_ascii=False,indent=2).encode())
    return TABLE

def upload_ckpt(final_ckpt,tag):
    pre=f"tts_checkpoints/omnivoice_yoruba/{tag}"; files=[]
    for root,_,fns in os.walk(final_ckpt):
        for fn in fns:
            lp=os.path.join(root,fn); files.append((lp,f"{pre}/{os.path.relpath(lp,final_ckpt)}"))
    with ThreadPoolExecutor(max_workers=16) as ex:
        list(ex.map(lambda t: s3.upload_file(t[0],BUCKET,t[1]),files))
    print(f"  uploaded {len(files)} ckpt files -> s3://{BUCKET}/{pre}/")

def run_one_budget(hours, seed):
    tag=f"{hours:g}h_s{seed}"; print(f"\n################  BUDGET {tag}  ################")
    train_rows,train_s=take_hours(rest,hours)
    assert train_rows, "no train rows"
    token_dir=os.path.join(WORK,"tokens",tag); output_dir=os.path.join(WORK,"exp",tag)
    os.makedirs(output_dir,exist_ok=True)
    tj=os.path.join(WORK,f"train_{tag}.jsonl"); dj=os.path.join(WORK,f"dev_{tag}.jsonl")
    write_manifest(train_rows,tj); write_manifest(dev_rows,dj)
    steps=max(50,round(EPOCHS_TARGET*len(train_rows)/CLIPS_PER_STEP))
    print(f"{tag}: {len(train_rows)} clips / {train_s/3600:.2f} h -> {steps} steps (~{EPOCHS_TARGET} epochs)")
    train_lst=stage0("train",tj,tag,token_dir); dev_lst=stage0("dev",dj,tag,token_dir)
    cp,dp,cfg=build_cfg(steps,train_lst,dev_lst,tag,output_dir,seed=seed)
    final_ckpt=stage1(cp,dp,output_dir,tag)
    # (multi-seed: checkpoint NOT uploaded — we keep only the scores)
    AGG=eval_budget(final_ckpt,tag,hours)
    shutil.rmtree(output_dir,ignore_errors=True); shutil.rmtree(token_dir,ignore_errors=True)  # free disk between runs
    print(f"################  {tag} DONE: tone_i2={AGG['tone_i2']:.3f} cer={AGG['cer']:.3f}  ################")
    return AGG
print("helpers defined — run the budget sections below (no edits needed).")

## 7. Sweep — section per budget (just **Run** each)
Each cell trains + evals one budget and keeps its own log. (`SWEEP_BUDGETS = [15.0, 5.0, 1.0]`.)

## 8. The data-efficiency curve

In [ ]:
import json, numpy as np, matplotlib.pyplot as plt
TABLE=json.loads(s3.get_object(Bucket=BUCKET,Key=ABLATION_KEY)["Body"].read())
ANCH=[(0.0,0.598,"zero-shot"),(28.5,0.633,"full")]   # known anchors (nb01 zero-shot; nb03/nb04 full)
pts=[(r["train_hours_actual"],r["tone_i2"],t) for t,r in TABLE["runs"].items()]
allp=sorted(pts+[(h,v,n) for h,v,n in ANCH], key=lambda x:x[0])
print(f"{'hours':>7} {'tone_i2':>8}  source")
for h,v,n in allp: print(f"{h:>7.2f} {v:>8.3f}  {n if n in ('zero-shot','full') else 'sweep'}")
xs=[h for h,_,_ in allp]; ys=[v for _,v,_ in allp]
plt.figure(figsize=(6,4)); plt.plot(xs,ys,"o-"); plt.axhline(1/3,ls=":",c="grey",label="chance 0.333")
plt.axhline(0.58,ls="--",c="green",label="native ~0.58")
for h,v,n in allp:
    if n in ("zero-shot","full"): plt.annotate(n,(h,v),textcoords="offset points",xytext=(4,4),fontsize=8)
plt.xlabel("clean Yoruba hours"); plt.ylabel("tone_i2"); plt.title("Data efficiency: tone vs hours")
plt.legend(); plt.tight_layout(); plt.savefig(f"{WORK}/curve.png",dpi=120)
s3.upload_file(f"{WORK}/curve.png",BUCKET,"tts_data/yoruba/omnivoice_ablation/curve.png"); plt.show()
print("\ncurve -> s3://%s/tts_data/yoruba/omnivoice_ablation/{data_efficiency.v1.json, curve.png}"%BUCKET)

## Run: every budget × every training seed (incremental S3 save)

In [ ]:
import shutil, subprocess
shutil.rmtree(os.path.join(WORK,"exp"),ignore_errors=True)   # wipe stale checkpoints from earlier runs
shutil.rmtree(os.path.join(WORK,"tokens"),ignore_errors=True)
print(subprocess.run(["df","-h",WORK],capture_output=True,text=True).stdout)
import json, datetime
MS={}                                              # (hours, seed) -> AGG
for seed in TRAIN_SEEDS:
    for hours in SWEEP_BUDGETS:
        print(f"\n========================  seed {seed} | {hours:g}h  ========================")
        MS[(hours,seed)]=run_one_budget(hours, seed)
        rec={f"{h:g}h|s{s}":v for (h,s),v in MS.items()}
        s3.put_object(Bucket=BUCKET,Key=MULTISEED_KEY,Body=json.dumps(
            {"meta":{"protocol":f"fix-epochs={EPOCHS_TARGET}, batch_tokens={BATCH_TOKENS}, lr=1e-5",
                     "train_seeds":TRAIN_SEEDS,"eval_seed":EVAL_SEED,"note":"data fixed per budget; only training seed varies",
                     "date":datetime.date.today().isoformat()},"runs":rec},ensure_ascii=False,indent=2).encode())
print("\nALL RUNS DONE -> s3://%s/%s"%(BUCKET,MULTISEED_KEY))

## Aggregate → TRAINING error bars + the decisive 5 h vs 15 h overlap check

In [ ]:
import numpy as np, matplotlib.pyplot as plt, IPython.display as ipd
budgets=sorted({h for h,_ in MS})
agg={}
for h in budgets:
    t=[MS[(h,s)]["tone_i2"] for s in TRAIN_SEEDS if (h,s) in MS]
    cc=[MS[(h,s)]["cer"] for s in TRAIN_SEEDS if (h,s) in MS]
    agg[h]=dict(mean=float(np.mean(t)),std=float(np.std(t,ddof=1)) if len(t)>1 else 0.0,
                cer=float(np.mean(cc)),n=len(t))
print(f"{'hours':>6} | {'tone_i2 mean ± std (TRAINING)':>30} | {'cer':>7} | n")
for h in budgets:
    a=agg[h]; print(f"{h:>6g} | {a['mean']:>14.3f} ± {a['std']:.3f}{'':>9} | {a['cer']:>7.3f} | {a['n']}")

# decisive check: do adjacent budgets' mean±std bands overlap?
print("\n5h vs 15h:", "OVERLAP -> curve shape is within training noise" if
      (5.0 in agg and 15.0 in agg and
       agg[5.0]['mean']+agg[5.0]['std'] >= agg[15.0]['mean']-agg[15.0]['std'])
      else "SEPARABLE -> the 5h->15h gain survives training variance")

xs=budgets; ys=[agg[h]['mean'] for h in xs]; es=[agg[h]['std'] for h in xs]
plt.figure(figsize=(6,4))
plt.errorbar(xs,ys,yerr=es,marker='o',capsize=4,lw=2,label='tone_i2 (mean ± std, TRAINING seeds)')
plt.axhline(0.58,ls='--',c='g',label='native ~0.58'); plt.axhline(0.333,ls=':',c='gray',label='chance')
plt.xlabel('clean Yoruba hours'); plt.ylabel('tone_i2'); plt.title('Data-efficiency with TRAINING error bars')
plt.legend(); plt.tight_layout()
p=f"{WORK}/curve_trainseeds.png"; plt.savefig(p,dpi=150)
s3.upload_file(p,BUCKET,"tts_data/yoruba/omnivoice_ablation/curve_trainseeds.png")
ipd.display(ipd.Image(p))
print("-> s3 .../omnivoice_ablation/curve_trainseeds.png  (this replaces the decode-only error bars on the poster)")